# Realistic LLM Use Case: When LLMs Actually Win
**SNIA DSN AI Stack Webinar Series — 2026**

This notebook demonstrates a realistic storage engineering task — classifying **unstructured incident report messages** into 6 categories — where LLM fine-tuning meaningfully outperforms traditional ML.

### Key Finding
TF-IDF + XGBoost achieves ~70–80%. SFT achieves ~85–92%. The gap comes from shared vocabulary across categories that only contextual understanding can disambiguate.

### Dataset
960 synthetic incident reports (120 per category, 8 templates each) with deliberately overlapping vocabulary.

## Two Modes

| Mode | What happens | Time |
|------|-------------|------|
| **Full Training** (Step 8A) | Train LoRA adapter from scratch on GPU | ~20 min on T4 |
| **Demo** (Step 8B) | Load pre-trained adapter from HuggingFace Hub | ~2 min |

## Prerequisites
- GPU runtime (T4 or A100) — Go to **Runtime > Change runtime type > T4 GPU**
- HuggingFace token (optional, for Step 8A model upload)

## How to Use
1. Run Steps 1–7 (data generation and baseline)
2. Choose **either** Step 8A (full training) **or** Step 8B (demo mode) — not both
3. Run Steps 9–12 (evaluation and analysis)

## Step 1: Prerequisites and HuggingFace Setup

Install all required packages, check GPU availability, and configure the HuggingFace token.

A HuggingFace token is needed for **Step 8A** to upload the trained model after training completes. If you only plan to use **Step 8B** (demo mode), you can leave the token blank.

To get your token:
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Create a new token with **write** access
3. Paste it in the `HF_TOKEN` variable below

In [ ]:
# Step 1: Prerequisites and HuggingFace Setup
!pip install -q torch transformers datasets accelerate peft trl scikit-learn xgboost matplotlib seaborn sentencepiece huggingface_hub

import torch
import time as _time

_notebook_start = _time.time()

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem:.1f} GB")
else:
    print("WARNING: GPU recommended for SFT training.")
    print("Go to Runtime > Change runtime type > T4 GPU")

# HuggingFace token (required for Step 8A model upload)
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = ""  # <-- Paste your token here

# Repository for the pre-trained adapter
HF_REPO = "gidft/smollm2-error-classifier-sft"

## Step 2: Define Error Categories

Six categories of storage infrastructure incident reports, each with 8 template variations at three difficulty tiers. Templates deliberately share vocabulary across categories to challenge TF-IDF approaches.

| Category | What it covers |
|----------|---------------|
| Drive Failure | SMART failures, media errors, device timeouts, RAID degradation |
| Network Issue | Link errors, path instability, fabric problems, connectivity failures |
| Capacity Warning | Pool usage, space exhaustion, thin provisioning, growth projections |
| Performance Degradation | Latency spikes, throughput drops, workload contention |
| Configuration Error | Misconfigs, incorrect parameters, routing errors |
| Firmware Bug | Vendor defects, version-specific bugs, false positive health alerts |

**Difficulty tiers per category:**
- ~3 **EASY**: category-specific language that TF-IDF can latch onto
- ~3 **MEDIUM**: shared vocabulary but context disambiguates
- ~2 **HARD**: cross-category causation or negation that confuses keyword-based classifiers

In [ ]:
# Step 2: Define Error Categories
import random
import numpy as np
from collections import Counter

random.seed(42)
np.random.seed(42)

# Shared variable definitions — identical across all categories to prevent variable-name leakage
SHARED_VARS = {
    "dev": ["sd{}", "nvme{}n1", "pd{}"],
    "node": lambda: f"node-{random.randint(1, 32):02d}",
    "node2": lambda: f"node-{random.randint(1, 32):02d}",
    "port": lambda: f"port{random.randint(0,7)}",
    "ctrl": lambda: f"ctrl-{random.choice(['a','b'])}-{random.randint(1,4)}",
    "pool": lambda: f"pool_{random.choice(['ssd','hdd','nvme','hybrid'])}_{random.randint(1,8)}",
    "vol": lambda: f"vol_{random.choice(['prod','db','app','logs'])}_{random.randint(1,20)}",
    "count": lambda: random.randint(3, 500),
    "hours": lambda: random.randint(1, 72),
    "mins": lambda: random.randint(1, 120),
    "pct": lambda: random.randint(50, 99),
    "ms": lambda: random.randint(10, 5000),
    "baseline_ms": lambda: random.randint(1, 10),
    "ver": lambda: f"{random.randint(3,12)}.{random.randint(0,9)}.{random.randint(0,9)}",
    "incident": lambda: f"INC-{random.randint(100000, 999999)}",
    "threshold": lambda: random.randint(70, 95),
    "ratio": lambda: round(random.uniform(1.2, 8.0), 1),
    "rate": lambda: round(random.uniform(10, 500), 1),
    "size_tb": lambda: round(random.uniform(1, 100), 1),
    "rg": lambda: f"rg{random.randint(0, 15)}",
    "host": lambda: f"host-{random.randint(1,50):02d}",
    "site": lambda: f"site-{random.choice(['east','west','central','dr'])}",
    "model_name": lambda: random.choice(["FlashArray//X70", "NetApp AFF A400", "PowerStore 9200T", "Unity XT 880F"]),
}

ERROR_CATEGORIES = {
    "Drive Failure": {
        "templates": [
            # EASY (3): category-specific language with some shared vocab
            "Device {dev} in RAID group {rg} reported {count} uncorrectable media errors over the past {hours} hours. "
            "SMART predictive failure flag is now set and latency on the device has exceeded {ms}ms. "
            "Controller {ctrl} has initiated an automatic rebuild on the hot spare.",

            "Drive {dev} on {node} experienced a timeout after {ms}ms with no response to the controller {ctrl}. "
            "The grown defect list increased by {count} entries this week and the SMART health status is now FAILING. "
            "Recommend immediate replacement before total device loss.",

            "RAID group {rg} on {node} degraded after device {dev} returned {count} consecutive I/O errors. "
            "Average response latency climbed to {ms}ms before the controller {ctrl} fenced the device. "
            "Rebuild to hot spare is in progress with an ETA of {hours} hours.",

            # MEDIUM (3): shared vocabulary, context disambiguates
            "Device {dev} errors on {node} triggered {count} iSCSI retransmissions on the fabric attached to {port}. "
            "Network metrics appear normal but packet traces confirm the retransmits originate at the device timeout layer. "
            "Root cause is physical media degradation, not a network fault.",

            "Drive {dev} running firmware version {ver} on controller {ctrl} logged {count} command timeouts in {hours} hours. "
            "Vendor analysis confirms the issue is a worn-out flash cell, not a firmware defect. "
            "The device should be replaced under warranty per incident {incident}.",

            "Volume {vol} on pool {pool} shows degraded throughput since device {dev} began returning slow responses. "
            "The controller {ctrl} reports average device latency of {ms}ms against a {baseline_ms}ms baseline. "
            "Replacing the failing drive is expected to restore pool performance.",

            # HARD (2): cross-category causation / negation
            "Cache destage on {node} failed repeatedly when writing to device {dev} in pool {pool}. "
            "Error logs show the cache contents could not be flushed because the target device returned media errors. "
            "This is a drive failure, not a cache or firmware issue. Replace device {dev} and the cache will destage normally.",

            "Replication from {site} stalled for {hours} hours with {count} pending write acknowledgments. "
            "Investigation shows device {dev} on the destination returned I/O errors that blocked replication commits. "
            "No configuration or network fault was found. The destination drive must be replaced to resume replication.",
        ],
        "vars": SHARED_VARS,
    },
    "Network Issue": {
        "templates": [
            # EASY (3): connectivity / link errors with some shared vocab
            "Link errors detected on {port} of {node} with {count} CRC failures in {mins} minutes. "
            "The controller {ctrl} triggered a multipath failover to the secondary path. "
            "Timeout threshold exceeded and the fabric requires cable or SFP inspection.",

            "Connectivity to {node} lost on {port} after {count} consecutive heartbeat failures. "
            "Controller {ctrl} marked all paths through that port as faulted. "
            "Latency on the remaining path increased to {ms}ms while the switch port is investigated.",

            "Fibre Channel port {port} on {node} logged {count} RSCN events causing fabric instability. "
            "Controller {ctrl} timeout threshold was breached during the link flap storm. "
            "Isolate the offending HBA to restore zone stability.",

            # MEDIUM (3): shared vocabulary, context disambiguates
            "After a firmware upgrade to version {ver} on the switch at {site}, path {port} on {node} began dropping packets. "
            "The issue is not a firmware defect on the storage controller but a network configuration change introduced during the upgrade. "
            "Rolling back the switch configuration resolved the packet loss.",

            "Buffer capacity on switch port {port} at {site} exceeded during replication between {node} and {node2}. "
            "The resulting packet drops resemble a capacity warning but the root cause is network buffer exhaustion, not storage pool space. "
            "Increasing the buffer allocation or enabling flow control will resolve the issue.",

            "Controller {ctrl} on {node} reported {count} timeout errors during peak traffic on {port}. "
            "Device {dev} health checks passed and no media errors were found. "
            "The timeouts correlate with network congestion on the inter-switch link, not with device faults.",

            # HARD (2): cross-category causation / negation
            "Performance degraded on volume {vol} with cache hit ratio dropping from {pct}% to {threshold}%. "
            "Investigation revealed the cache misses were caused by path instability on {port} of {node} that forced repeated read retries. "
            "Resolving the network path issue restored cache efficiency and performance.",

            "Device {dev} on {node} appeared to timeout at {ms}ms but the controller {ctrl} confirmed no device errors. "
            "Packet capture shows the timeout originated from a congested network hop between {site} and the array. "
            "This is a network issue, not a drive failure. The device is healthy.",
        ],
        "vars": SHARED_VARS,
    },
    "Capacity Warning": {
        "templates": [
            # EASY (3): pool usage, space projections
            "Storage pool {pool} usage has reached {pct}% of its {size_tb}TB allocation. "
            "At the current growth rate the pool will exceed the {threshold}% threshold within {hours} hours. "
            "Thin provisioning overcommit ratio is {ratio}x and automatic alerts have been sent to the storage team.",

            "Volume {vol} on pool {pool} has consumed {pct}% of its quota. "
            "Snapshot reserve alone accounts for {size_tb}TB across {count} retained snapshots. "
            "The oldest snapshots should be reviewed and deleted to reclaim space before the volume goes read-only.",

            "Aggregate capacity on {node} shows {pct}% utilization with {count} volumes competing for the remaining space. "
            "Pool {pool} is the largest consumer at {size_tb}TB. "
            "A capacity expansion or volume rebalance across nodes is recommended within {hours} hours.",

            # MEDIUM (3): shared vocabulary, context disambiguates
            "Write operations to volume {vol} on pool {pool} are being throttled to {rate} IOPS. "
            "The throttling is not a performance issue but a protective measure because the pool has exceeded {pct}% capacity. "
            "Adding storage to the pool will remove the throttle and restore normal write rates.",

            "Device {dev} on {node} returned write errors with error code indicating no space remaining on pool {pool}. "
            "This is not a device failure; the drive hardware passed all diagnostic checks. "
            "The errors will stop once pool {pool} capacity is expanded beyond {pct}%.",

            "Replication from {site} to the DR site is falling behind with {count} pending writes. "
            "The replication lag is caused by the destination pool {pool} reaching {pct}% capacity and rejecting new writes. "
            "Free up space on the destination pool to resume replication.",

            # HARD (2): cross-category causation / negation
            "Controller {ctrl} on {node} timed out while servicing thin-provisioned volume {vol}. "
            "No firmware defect or device error was found. The timeout occurred because pool {pool} hit {pct}% capacity and the controller "
            "could not allocate new blocks. Expanding the pool is the correct resolution, not a controller or drive replacement.",

            "Latency on volume {vol} degraded from {baseline_ms}ms to {ms}ms over the past {hours} hours. "
            "The degradation correlates exactly with pool {pool} exceeding the {threshold}% capacity threshold. "
            "No workload change or hardware fault is present. Freeing space in the pool will restore latency to baseline.",
        ],
        "vars": SHARED_VARS,
    },
    "Performance Degradation": {
        "templates": [
            # EASY (3): latency spikes, throughput drops, workload contention
            "Latency spike detected on volume {vol} on {node} with p99 latency jumping from {baseline_ms}ms to {ms}ms. "
            "Queue depth is averaging {count} during a workload contention event on controller {ctrl}. "
            "The spike is caused by a noisy neighbor workload and no hardware fault is involved.",

            "Throughput on pool {pool} dropped from {rate} MB/s to {baseline_ms} MB/s over the last {mins} minutes. "
            "Controller {ctrl} shows {pct}% utilization with elevated cache miss rates. "
            "The throughput drop correlates with a batch analytics job competing for I/O bandwidth.",

            "Volume {vol} on {node} experiencing sustained read latency of {ms}ms against a baseline of {baseline_ms}ms. "
            "The degradation began when {count} additional workloads were scheduled on the same controller {ctrl}. "
            "Spreading workloads across controllers is recommended to restore performance.",

            # MEDIUM (3): shared vocabulary, context disambiguates
            "Controller {ctrl} on {node} running firmware version {ver} shows degraded latency of {ms}ms. "
            "The firmware version is unchanged from before the issue started and vendor has confirmed no defect at this version. "
            "Root cause is workload contention between {count} volumes sharing the controller.",

            "Pool {pool} on {node} has reached {pct}% utilization but the capacity threshold has not been breached. "
            "Latency increased to {ms}ms due to I/O contention at high utilization, not space exhaustion. "
            "Rebalancing workloads across pools will improve response times without adding capacity.",

            "Cache hit ratio on controller {ctrl} dropped from {threshold}% to {pct}% after a workload profile change. "
            "All network paths on {port} of {node} are healthy and no device errors are present. "
            "The degradation is caused by a working set that exceeds the cache capacity, not by any infrastructure fault.",

            # HARD (2): cross-category causation / negation
            "Device {dev} response time on {node} degraded to {ms}ms but controller {ctrl} diagnostics confirm the device is healthy. "
            "The bottleneck is at the controller level where {count} volumes are contending for processing resources. "
            "This is a performance problem, not a drive failure. Redistribute volumes to resolve the contention.",

            "Write operations to volume {vol} on pool {pool} failed intermittently during peak load on {node}. "
            "No hardware faults, network errors, or capacity limits were found. "
            "The failures are caused by QoS throttling at {rate} IOPS on controller {ctrl} under excessive concurrent load.",
        ],
        "vars": SHARED_VARS,
    },
    "Configuration Error": {
        "templates": [
            "Multipath on {host} is misconfigured with both paths through {port} routed to the same controller {ctrl}. Failover testing failed because no independent path exists. The incorrect path configuration must be corrected before the next maintenance window.",
            "Tiering policy on pool {pool} has an incorrect threshold set to {threshold}% instead of the required {pct}%. This misconfiguration is keeping {size_tb}TB of hot data on the slow tier. Correcting the tiering threshold will resolve the data placement issue on controller {ctrl}.",
            "Connectivity errors on fabric path {port} between {node} and {node2} caused {count} retransmissions in {hours} hours. Network infrastructure tests passed and no cable or SFP faults were found. The retransmissions are caused by a configuration mismatch in the multipath routing table that was introduced during maintenance.",
            "Pool {pool} on {node} triggered a capacity threshold alert at {pct}% utilization with a projected growth rate of {size_tb}TB per week. However, actual usage is only {threshold}% and the thin provisioning overcommit ratio is within normal limits. The alert is a false positive caused by an incorrectly configured monitoring threshold, not a real capacity issue.",
            "Device {dev} on {node} generated {count} errors and controller {ctrl} flagged the device as degraded, initiating a rebuild. All SMART diagnostics and media scans on the device returned healthy. The errors are caused by an incorrect I/O queue depth setting applied during a recent configuration change per incident {incident}.",
            "Connectivity to {node} lost on {port} with path failover failing to the secondary controller {ctrl}. Fabric instability was initially suspected but switch diagnostics and cable checks returned clean. Root cause is a misconfigured ALUA target port group setting that routes all traffic to a single path. Correcting the path priority configuration will restore multipath functionality.",
            "Latency on volume {vol} degraded by {ratio}x on {node} with throughput dropping to {rate} MB/s and cache hit ratio falling to {pct}%. No workload change, capacity constraint, or firmware defect was identified. Investigation found that a QoS bandwidth limit was incorrectly applied to volume {vol} during a recent policy change. Removing the erroneous QoS configuration restored performance to baseline.",
            "Replication lag between {site} and the DR site reached {hours} hours with {count} pending writes and destination pool {pool} reporting {pct}% capacity. Network path verification between sites returned clean and the destination pool has adequate free space. The replication stall is caused by a misconfigured replication schedule that triggers {count} concurrent streams instead of staggered batches. Fixing the schedule configuration will resume normal replication flow."
        ],
        "vars": SHARED_VARS,
    },
    "Firmware Bug": {
        "templates": [
            "Vendor advisory for firmware version {ver} on {model_name} confirms a known defect in the cache coherency module. Controller {ctrl} on {node} is affected and {count} volumes are at risk. Upgrade to the patched firmware version per incident {incident} to resolve the defect.",
            "Storage OS version {ver} on {node} contains a memory leak in the deduplication engine. Resident memory is growing at {rate}MB per hour and controller {ctrl} will run out of memory in {hours} hours. Vendor has released a hotfix. Apply the patched version as described in incident {incident}.",
            "Latency spike detected on volume {vol} with throughput dropping to {rate} MB/s and queue depth averaging {count} on {node}. No workload contention or capacity constraint was identified. Issue reproduces only on controllers running firmware version {ver}, confirming a firmware defect in the I/O scheduler.",
            "Device {dev} on {node} reporting {count} failed reads and timeout errors at {ms}ms under moderate load. All SMART diagnostics passed and no media degradation was found on any device in pool {pool}. Vendor root cause analysis traced the failed reads to a firmware bug in the error-handling routine of version {ver}.",
            "Cache hit rate on controller {ctrl} dropped from {threshold}% to {pct}% and response time degraded to {ms}ms across pool {pool}. Workload analysis shows no change in I/O profile or contention levels. The cache eviction algorithm in firmware version {ver} has a confirmed defect that vendor is patching per incident {incident}.",
            "Device {dev} on {node} was marked degraded by controller {ctrl} and a rebuild to the hot spare was initiated. However, media scans returned zero errors and the device passed all SMART diagnostics. The health monitoring module in firmware version {ver} has a known false-positive bug; no actual drive failure occurred. Apply the vendor patch per incident {incident} to stop the spurious device fencing.",
            "Sustained throughput degradation on {node} with latency rising from {baseline_ms}ms to {ms}ms and cache miss rate exceeding {pct}%. Queue depth on controller {ctrl} reached {count} during normal workload with no noisy neighbor activity detected. The bottleneck was traced to a race condition in firmware version {ver} that serializes concurrent cache flushes. Vendor confirmed the defect and a patch is pending under incident {incident}.",
            "Retransmissions on fabric port {port} of {node} spiked to {count} per minute with device {dev} timeout errors at {ms}ms. No hardware fault was detected on the drive and network path diagnostics returned clean. The retransmissions originate from a firmware defect in version {ver} that miscalculates the SCSI timeout window. Vendor advisory {incident} documents this as a known firmware bug with a patch available."
        ],
        "vars": SHARED_VARS,
    },
}

CATEGORIES = list(ERROR_CATEGORIES.keys())
print(f"Error categories: {CATEGORIES}")
print(f"Templates per category: {[len(v['templates']) for v in ERROR_CATEGORIES.values()]}")

## Step 3: Generate Synthetic Incident Report Dataset

Generate 960 incident reports (120 per category, 8 templates each = 15 per template) by filling templates with randomized variable values. Each report is realistic enough to challenge a classifier.

In [ ]:
# Step 3: Generate Synthetic Incident Report Dataset
import re

def fill_template(template, var_defs):
    """Fill a template with random variable values."""
    filled = template
    needed = set(re.findall(r'\{(\w+)\}', template))
    values = {}
    for var_name in needed:
        if var_name in var_defs:
            v = var_defs[var_name]
            if callable(v):
                values[var_name] = v()
            elif isinstance(v, list):
                choice = random.choice(v)
                if '{}' in choice:
                    choice = choice.format(chr(random.randint(97, 122)))
                values[var_name] = choice
            else:
                values[var_name] = v
        else:
            values[var_name] = f"UNKNOWN_{var_name}"
    return filled.format(**values)

def generate_log_message(category, template_idx=None):
    """Generate a single incident report for a given category."""
    cat_def = ERROR_CATEGORIES[category]
    if template_idx is not None:
        template = cat_def["templates"][template_idx]
    else:
        template = random.choice(cat_def["templates"])
    return fill_template(template, cat_def["vars"])

# Generate 960 samples (120 per category, 15 per template)
samples = []
for cat in CATEGORIES:
    n_templates = len(ERROR_CATEGORIES[cat]["templates"])
    for t_idx in range(n_templates):
        for _ in range(15):
            msg = generate_log_message(cat, template_idx=t_idx)
            samples.append({"text": msg, "label": cat})

random.shuffle(samples)
print(f"Generated {len(samples)} incident reports")
print(f"Categories: {dict(Counter(s['label'] for s in samples))}")

## Step 4: Explore the Data

Display 2 examples per category to see the variety of phrasing. Notice how the same error type can be expressed with completely different vocabulary and how different categories share terminology like "latency," "timeout," "controller," and "firmware."

In [ ]:
# Step 4: Explore the Data
for cat in CATEGORIES:
    cat_samples = [s for s in samples if s['label'] == cat][:2]
    print(f"\n{'='*70}")
    print(f"  {cat}")
    print(f"{'='*70}")
    for i, s in enumerate(cat_samples):
        print(f"\n  Example {i+1}:")
        # Word-wrap at ~70 chars
        words = s['text'].split()
        line = "    "
        for w in words:
            if len(line) + len(w) + 1 > 74:
                print(line)
                line = "    " + w
            else:
                line += " " + w if line.strip() else "    " + w
        if line.strip():
            print(line)
print()

## Step 5: Baseline — TF-IDF + XGBoost

First, we establish a traditional ML baseline. TF-IDF converts text to word-frequency features, then XGBoost classifies. This is a strong baseline for text classification — but it treats each message as a bag of words, ignoring word order and context.

In [ ]:
# Step 5: Baseline — TF-IDF + XGBoost
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import time

# Prepare data
texts = [s['text'] for s in samples]
labels = [s['label'] for s in samples]

le = LabelEncoder()
y = le.fit_transform(labels)

# Train/test split
texts_train, texts_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
print("Vectorizing with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(texts_train)
X_test_tfidf = tfidf.transform(texts_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Feature matrix: {X_train_tfidf.shape}")

# Train XGBoost
print("\nTraining XGBoost on TF-IDF features...")
t0 = time.time()
xgb_text = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                          eval_metric='mlogloss')
xgb_text.fit(X_train_tfidf, y_train)
xgb_train_time = time.time() - t0

# Evaluate
xgb_preds = xgb_text.predict(X_test_tfidf)
xgb_accuracy = accuracy_score(y_test, xgb_preds)

print(f"\nTraining time: {xgb_train_time:.2f}s")
print(f"Accuracy: {xgb_accuracy:.1%}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

## Step 6: Why XGBoost Struggles Here

The accuracy above is decent but limited. Three common failure modes:

1. **Vocabulary overlap:** "latency," "timeout," "controller," and "firmware" appear in 3+ categories — TF-IDF cannot distinguish context
2. **Negation context:** "No device errors — timeouts caused by space exhaustion" contains drive-failure vocabulary but the meaning is capacity warning
3. **Cross-category causation:** "Cache destage failed to device" sounds like firmware/performance but is actually a drive failure causing the cache issue

An LLM processes the full message as a sequence, understanding context, negation, and causal relationships rather than just word frequency.

## Step 7: Prepare SFT Training Data

Format each sample as a prompt/completion pair for supervised fine-tuning. The prompt includes the full incident report and asks for a classification; the completion is the category label. This is the same format used in the Post-Training Pipeline.

In [ ]:
# Step 7: Prepare SFT Training Data
from datasets import Dataset
from sklearn.model_selection import train_test_split as tts2

def format_for_sft(sample):
    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    completion = f" {sample['label']}"
    return {"prompt": prompt, "completion": completion}

all_formatted = [format_for_sft(s) for s in samples]

# Split formatted data the same way as Step 5
random.seed(42)
np.random.seed(42)
fmt_train, fmt_test = tts2(all_formatted, test_size=0.2, random_state=42,
                            stratify=[s['label'] for s in samples])

train_dataset = Dataset.from_dict({
    "prompt": [s["prompt"] for s in fmt_train],
    "completion": [s["completion"] for s in fmt_train],
})

print(f"SFT training samples: {len(train_dataset)}")
print(f"\nExample training input:")
print(f"  Prompt (first 300 chars): {fmt_train[0]['prompt'][:300]}...")
print(f"  Completion: {fmt_train[0]['completion']}")

## Step 8A: Full Training Mode (skip for demo)

Train a LoRA adapter on SmolLM2-360M from scratch. This takes ~20 minutes on a T4 GPU. After training, the adapter is saved locally and automatically uploaded to the HuggingFace Hub if `HF_TOKEN` was set in Step 1.

**Skip this cell** if you want a quick demo — use Step 8B instead to load a pre-trained adapter.

In [ ]:
# Step 8A: Full Training Mode (~20 min on T4)
# SKIP THIS CELL for demo mode — use Step 8B instead

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for training. Go to Runtime > Change runtime type > T4 GPU")

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

set_seed(42)

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA configuration — same settings as Post-Training Pipeline
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Training configuration
training_args = SFTConfig(
    output_dir="/tmp/sft_error_logs",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("\nStarting SFT training...")
t0 = time.time()
trainer.train()
sft_train_time = time.time() - t0
print(f"\nTraining completed in {sft_train_time/60:.1f} minutes")

# Save adapter locally
model.save_pretrained("/tmp/sft_error_logs/final_adapter")
tokenizer.save_pretrained("/tmp/sft_error_logs/final_adapter")
print("Adapter saved to /tmp/sft_error_logs/final_adapter")

# Upload to HuggingFace Hub
if HF_TOKEN:
    import os
    cache_path = os.path.expanduser("~/.cache/huggingface/token")
    if os.path.exists(cache_path):
        os.remove(cache_path)
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Uploaded to {HF_REPO}")
else:
    print("No HF_TOKEN set in Step 1 — skipping upload.")

## Step 8B: Demo Mode — Load Pre-trained Adapter

Load a pre-trained LoRA adapter from the HuggingFace Hub. This skips the ~20 minute training step and lets you jump straight to evaluation. The adapter was trained with the exact same configuration as Step 8A.

In [ ]:
# Step 8B: Demo Mode — Load Pre-trained Adapter from HuggingFace Hub
# USE THIS CELL instead of Step 8A for a quick demo

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for inference. Go to Runtime > Change runtime type > T4 GPU")

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading base model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading pre-trained adapter: {HF_REPO}...")
model = PeftModel.from_pretrained(base_model, HF_REPO)
print("Adapter loaded successfully.")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {trainable:,} trainable / {total:,} total")

# Mark that we skipped training
sft_train_time = 0

## Step 9: Evaluate the SFT Model

Run inference on the test set and compare predictions against ground truth. Each sample is formatted with the same prompt template used during training.

In [ ]:
# Step 9: Evaluate the SFT Model
model.eval()

# Reconstruct test set
_, test_samples_eval = tts2(samples, test_size=0.2, random_state=42,
                             stratify=[s['label'] for s in samples])

sft_preds = []
sft_correct = 0
total_test = len(test_samples_eval)

print(f"Evaluating SFT model on {total_test} test samples...")
for idx, sample in enumerate(test_samples_eval):
    if (idx + 1) % 20 == 0 or idx == 0:
        print(f"  Progress: {idx+1}/{total_test}")

    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Check if any category name appears in the output
    predicted = None
    for cat in CATEGORIES:
        if cat.lower() in generated.lower():
            predicted = cat
            break

    is_correct = predicted == sample['label']
    if is_correct:
        sft_correct += 1
    sft_preds.append({
        'true': sample['label'],
        'predicted': predicted or generated[:50],
        'raw_output': generated[:100],
        'correct': is_correct,
    })

sft_accuracy = sft_correct / len(test_samples_eval)
print(f"\nSFT Accuracy: {sft_accuracy:.1%} ({sft_correct}/{len(test_samples_eval)})")

# Per-category breakdown
print(f"\nPer-Category Results:")
print("-" * 50)
for cat in CATEGORIES:
    cat_preds = [p for p in sft_preds if p['true'] == cat]
    cat_correct = sum(1 for p in cat_preds if p['correct'])
    cat_total = len(cat_preds)
    pct = cat_correct / cat_total if cat_total > 0 else 0
    bar = "#" * int(pct * 10) + "." * (10 - int(pct * 10))
    print(f"  {cat:<25s} {bar} {cat_correct}/{cat_total} ({pct:.0%})")

## Step 10: Head-to-Head Comparison

Side-by-side charts comparing TF-IDF + XGBoost vs SFT fine-tuning on overall accuracy and per-category accuracy, plus a summary table with training time, model size, and qualitative differences.

In [ ]:
# Step 10: Head-to-Head Comparison
import matplotlib.pyplot as plt
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy comparison
models = ['TF-IDF + XGBoost', 'SFT (SmolLM2-360M)']
accs = [xgb_accuracy, sft_accuracy]
colors = ['#f97316', '#3b82f6']
bars = ax1.bar(models, accs, color=colors, alpha=0.8, width=0.5)
ax1.set_ylabel('Accuracy')
ax1.set_title('Text Classification Accuracy')
ax1.set_ylim(0, 1.1)
for bar, val in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.1%}', ha='center', fontweight='bold', fontsize=13)

# Right: Per-category comparison
xgb_cat_acc = {}
sft_cat_acc = {}
for cat in CATEGORIES:
    # XGBoost
    cat_mask = [le.classes_[y_test[i]] == cat for i in range(len(y_test))]
    cat_indices = [i for i, m in enumerate(cat_mask) if m]
    if cat_indices:
        xgb_cat_acc[cat] = sum(1 for i in cat_indices if xgb_preds[i] == y_test[i]) / len(cat_indices)
    # SFT
    cat_sft = [p for p in sft_preds if p['true'] == cat]
    if cat_sft:
        sft_cat_acc[cat] = sum(1 for p in cat_sft if p['correct']) / len(cat_sft)

x = np.arange(len(CATEGORIES))
width = 0.35
ax2.bar(x - width/2, [xgb_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='TF-IDF + XGBoost', color='#f97316', alpha=0.8)
ax2.bar(x + width/2, [sft_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='SFT', color='#3b82f6', alpha=0.8)
ax2.set_ylabel('Accuracy')
ax2.set_title('Per-Category Accuracy')
ax2.set_xticks(x)
ax2.set_xticklabels(CATEGORIES, rotation=35, ha='right', fontsize=8)
ax2.legend()
ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 70)
print("  COMPARISON: TF-IDF + XGBoost vs SFT Fine-Tuning")
print("=" * 70)
if sft_train_time > 0:
    sft_time_str = f'{sft_train_time/60:.1f} min'
else:
    sft_time_str = 'Pre-trained (skipped)'

comparison_data = {
    'Metric': ['Accuracy', 'Training Time', 'Model Size', 'Hardware', 'Handles Novel Phrasing', 'Contextual Understanding'],
    'TF-IDF + XGBoost': [f'{xgb_accuracy:.1%}', f'{xgb_train_time:.1f}s', '~5 MB', 'CPU', 'Limited', 'None (bag of words)'],
    'SFT (SmolLM2-360M)': [f'{sft_accuracy:.1%}', sft_time_str, '~700 MB', 'GPU', 'Good', 'Full sequence modeling'],
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("=" * 70)

## Step 11: Error Analysis

Examine cases where the SFT model succeeds and TF-IDF + XGBoost fails. This highlights the LLM's advantage in understanding contextual meaning.

In [ ]:
# Step 11: Error Analysis — Where does the LLM win?
print("Cases where SFT succeeds and TF-IDF + XGBoost fails:")
print("=" * 70)

# Align test samples between the two models
sft_test = test_samples_eval
count = 0
for i, (sample, sft_pred) in enumerate(zip(sft_test, sft_preds)):
    if i >= len(y_test):
        break
    xgb_pred_label = le.classes_[xgb_preds[i]] if i < len(xgb_preds) else None
    xgb_wrong = xgb_pred_label != sample['label']
    sft_right = sft_pred['correct']
    
    if xgb_wrong and sft_right and count < 4:
        count += 1
        print(f"\n  Example {count}:")
        print(f"  True label:    {sample['label']}")
        print(f"  XGBoost pred:  {xgb_pred_label} X")
        print(f"  SFT pred:      {sft_pred['predicted']} OK")
        text = sample['text'][:150]
        print(f"  Log message:   {text}...")

if count == 0:
    print("  (No clear examples in this run -- both models may have similar error patterns)")

print(f"\n\nSummary: SFT's advantage comes from understanding the *meaning* of incident reports,")
print(f"not just keyword frequency. This matters most when:")
print(f"  - Different categories share vocabulary (latency, timeout, controller, firmware)")
print(f"  - Negation changes meaning (no device errors, not a firmware defect)")
print(f"  - Cross-category causation requires reasoning (cache failed because drive failed)")

## Step 12: Conclusion — A Framework for Choosing Your Approach

| Characteristic | Use Traditional ML | Use LLM Fine-Tuning |
|---|---|---|
| **Input format** | Structured numbers, tables | Unstructured text, natural language |
| **Feature engineering** | Clear, known features | Features hard to extract manually |
| **Vocabulary** | N/A or controlled | Variable, natural language |
| **Context dependency** | Low — features are independent | High — meaning depends on context |
| **Training resources** | CPU, seconds-minutes | GPU, minutes-hours |
| **Model size** | KBs-MBs | Hundreds of MBs |
| **Interpretability** | High (feature importance) | Low (black box) |

**The two notebooks together give you a decision framework:**
- **Structured numeric data** (I/O metrics, sensor readings, performance counters) -> Random Forest / XGBoost
- **Unstructured text** (logs, error messages, incident reports, documentation) -> LLM fine-tuning

**Neither approach is universally better.** The right choice depends on your data, not on what's trending.

Return to the [Post-Training Pipeline](Post_Training_Pipeline.ipynb) to run the full SFT -> DPO -> GRPO pipeline, or see the [Traditional ML Comparison](Traditional_ML_Comparison.ipynb) for the structured data perspective.